## Experiment A0: local fixed-point dynamics¶
This notebook provides an end-to-end reproduction of Experiment A0 and its corresponding plot (Figure 1 in https://arxiv.org/abs/2609.04338).

The released CSV files in `data/` are the outputs from the original runs used in the paper. With `RERUN_EXPERIMENT = False`, the notebook reproduces the paper analysis and figure from those released outputs. With `RERUN_EXPERIMENT = True`, it regenerates all 420 runs from the released seeds, writes fresh outputs to `reproduced/`, checks them against the released outputs, and then performs the analysis and generates the figure from the fresh rerun.

We did not retain the exact NumPy and BLAS/LAPACK versions used for the original runs. Consequently, the lowest-residual floating-point behaviour can differ slightly across machines, especially in the two high-$p$ runs also discussed in the paper.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sys

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)

# If the notebook is run from the A0 release directory, these paths work as written.
DATA_DIR = Path("data")
OUT_DIR = Path("reproduced")
OUT_DIR.mkdir(exist_ok=True)

m, n = 100, 20
tol = 1e-11
max_iter = 5000

p_grid = [3, 4, 6, 8, 11, 12, 16, 17, 24, 32, 48, 54, 61, 64]
final_seeds = [
    22, 24, 29, 63, 89, 117, 143, 173, 176, 181,
    207, 233, 271, 319, 364, 411, 478, 523, 571, 607,
    611, 618, 674, 729, 803, 846, 917, 1009, 1183, 1427,
]

primary_window = (1e-8, 1e-4)
sensitivity_window_1 = (1e-7, 1e-4)
sensitivity_window_2 = (1e-8, 1e-5)

print(f"{len(final_seeds)} matrices x {len(p_grid)} p-values = {len(final_seeds) * len(p_grid)} runs")

## Lewis iteration and local prediction

In [ ]:
def lewis_update(A, w, p):
    gamma = 1 - 2 / p
    B = (w ** (gamma / 2))[:, None] * A
    Q, _ = np.linalg.qr(B, mode="reduced")
    return np.sum(Q**2, axis=1)


def run_lewis(A, p, tol=tol, max_iter=max_iter):
    m_local, n_local = A.shape
    w = np.full(m_local, n_local / m_local)

    residual_history = []
    weight_history = [w.copy()]

    for _ in range(max_iter):
        w_new = lewis_update(A, w, p)
        residual = np.max(np.abs(np.log(w_new / w)))

        residual_history.append(residual)
        weight_history.append(w_new.copy())

        if residual < tol:
            return w_new, np.array(residual_history), np.array(weight_history)

        w = w_new.copy()

    raise RuntimeError(f"Lewis iteration did not converge for p={p}")


def estimate_mu_obs(residual_history, fit_lo, fit_hi):
    mask = (residual_history >= fit_lo) & (residual_history <= fit_hi)
    t_fit = np.where(mask)[0]
    R_fit = residual_history[mask]

    if len(t_fit) < 3:
        return np.nan, len(t_fit), np.nan

    slope, intercept = np.polyfit(t_fit, np.log(R_fit), deg=1)
    mu_obs = np.exp(slope)

    fitted = intercept + slope * t_fit
    actual = np.log(R_fit)
    ss_res = np.sum((actual - fitted) ** 2)
    ss_tot = np.sum((actual - actual.mean()) ** 2)
    r_squared = 1 - ss_res / ss_tot

    return mu_obs, len(t_fit), r_squared


def local_factor(A, w, p):
    gamma = 1 - 2 / p
    B = (w ** (gamma / 2))[:, None] * A
    Q, _ = np.linalg.qr(B, mode="reduced")

    P = Q @ Q.T
    d = np.diag(P)

    # Symmetric matrix similar to D^{-1}(P o P), where D = Diag(diag(P)).
    S = (P**2) / (np.sqrt(d)[:, None] * np.sqrt(d)[None, :])
    lambda_min = np.linalg.eigvalsh(S)[0]

    return gamma * (1 - lambda_min)


def predicted_factor_and_diagnostics(A, w, p):
    gamma = 1 - 2 / p
    B = (w ** (gamma / 2))[:, None] * A
    Q, _ = np.linalg.qr(B, mode="reduced")

    P = Q @ Q.T
    d = np.diag(P)

    fixed_point_defect = np.max(np.abs(np.log(d / w)))
    projection_defect = np.linalg.norm(P @ P - P, ord=2)

    S = (P**2) / (np.sqrt(d)[:, None] * np.sqrt(d)[None, :])
    eigenvalues = np.linalg.eigvalsh(S)
    lambda_min = eigenvalues[0]
    lambda_max = eigenvalues[-1]

    return {
        "mu_pred": gamma * (1 - lambda_min),
        "lambda_min_K": lambda_min,
        "lambda_max_K": lambda_max,
        "fixed_point_defect": fixed_point_defect,
        "projection_defect": projection_defect,
    }


def analyze_one_run(A, p):
    w_hat, residual_history, weight_history = run_lewis(A, p)
    pred = predicted_factor_and_diagnostics(A, w_hat, p)

    mu_primary, n_primary, r2_primary = estimate_mu_obs(
        residual_history, *primary_window
    )
    mu_sens1, n_sens1, r2_sens1 = estimate_mu_obs(
        residual_history, *sensitivity_window_1
    )
    mu_sens2, n_sens2, r2_sens2 = estimate_mu_obs(
        residual_history, *sensitivity_window_2
    )

    last_mu_preds = np.array([
        local_factor(A, w, p) for w in weight_history[-5:]
    ])

    mu_pred = pred["mu_pred"]

    summary = {
        "p": p,
        "iterations": len(residual_history),
        "mu_pred": mu_pred,
        "mu_obs": mu_primary,
        "abs_gap": abs(mu_pred - mu_primary),
        "rel_gap": abs(mu_pred - mu_primary) / mu_pred,
        "mu_obs_sens1": mu_sens1,
        "mu_obs_sens2": mu_sens2,
        "fit_points": n_primary,
        "fit_R2": r2_primary,
        "sens1_points": n_sens1,
        "sens1_R2": r2_sens1,
        "sens2_points": n_sens2,
        "sens2_R2": r2_sens2,
        "fixed_point_defect": pred["fixed_point_defect"],
        "projection_defect": pred["projection_defect"],
        "mu_pred_stability": last_mu_preds.max() - last_mu_preds.min(),
        "lambda_min_K": pred["lambda_min_K"],
        "lambda_max_K": pred["lambda_max_K"],
    }

    return summary, residual_history

## Rerun all 420 experiments

In [ ]:
RERUN_EXPERIMENT = True

if RERUN_EXPERIMENT:
    all_results = []
    all_trajectories = []

    for seed_value in final_seeds:
        rng = np.random.default_rng(seed_value)
        A = rng.standard_normal((m, n))

        rank = np.linalg.matrix_rank(A)
        condition_number = np.linalg.cond(A)

        if rank != n:
            raise RuntimeError(
                f"Seed {seed_value} produced rank {rank}, expected {n}"
            )

        for p_value in p_grid:
            summary, residuals = analyze_one_run(A, p_value)
            summary["seed"] = seed_value
            summary["matrix_rank"] = rank
            summary["condition_number"] = condition_number
            all_results.append(summary)

            for t, residual in enumerate(residuals):
                all_trajectories.append({
                    "seed": seed_value,
                    "p": p_value,
                    "iteration": t,
                    "residual": residual,
                })

        print(f"finished seed {seed_value}")

    rerun_summary = pd.DataFrame(all_results)
    rerun_trajectories = pd.DataFrame(all_trajectories)

    rerun_p_summary = (
        rerun_summary.groupby("p", as_index=False)
        .agg(
            mean_mu_pred=("mu_pred", "mean"),
            mean_mu_obs=("mu_obs", "mean"),
            sd_mu_obs=("mu_obs", "std"),
            mean_mu_obs_sens1=("mu_obs_sens1", "mean"),
            sd_mu_obs_sens1=("mu_obs_sens1", "std"),
            mean_rel_gap=("rel_gap", "mean"),
            max_rel_gap=("rel_gap", "max"),
        )
    )

    rerun_summary.to_csv(OUT_DIR / "A0_run_summary.csv", index=False)
    rerun_trajectories.to_csv(
        OUT_DIR / "A0_residual_trajectories.csv", index=False
    )
    rerun_p_summary.to_csv(OUT_DIR / "A0_p_summary.csv", index=False)

    print("saved fresh rerun outputs in", OUT_DIR)
else:
    print("Skipping the full rerun. Set RERUN_EXPERIMENT = True to run it.")


## Check the fresh rerun against the released results

In [ ]:
released_summary = pd.read_csv(DATA_DIR / "A0_run_summary.csv")
released_trajectories = pd.read_csv(DATA_DIR / "A0_residual_trajectories.csv")
released_p_summary = pd.read_csv(DATA_DIR / "A0_p_summary.csv")

if RERUN_EXPERIMENT:
    old = released_summary.sort_values(["seed", "p"]).reset_index(drop=True)
    new = rerun_summary.sort_values(["seed", "p"]).reset_index(drop=True)

    same_cases = old[["seed", "p"]].equals(new[["seed", "p"]])
    assert same_cases
    assert len(new) == 420
    assert np.isfinite(new.select_dtypes(include=[np.number]).to_numpy()).all()

    final_residuals = (
        rerun_trajectories.sort_values(["seed", "p", "iteration"])
        .groupby(["seed", "p"], as_index=False)
        .tail(1)
    )
    assert (final_residuals["residual"] < tol).all()

    print("same 420 (seed, p) cases:", same_cases)
    print("all fresh runs reached the stopping threshold:", True)
    print("released trajectory rows:", len(released_trajectories))
    print("fresh-rerun trajectory rows:", len(rerun_trajectories))
    print("maximum |mu_pred(rerun) - mu_pred(released)|:",
          np.max(np.abs(new["mu_pred"] - old["mu_pred"])))
    print("maximum |mu_obs(rerun) - mu_obs(released)|:",
          np.max(np.abs(new["mu_obs"] - old["mu_obs"])))
    print("runs with a different stopping iteration:",
          np.sum(new["iterations"] != old["iterations"]))

    old_p = released_p_summary.sort_values("p").reset_index(drop=True)
    new_p = rerun_p_summary.sort_values("p").reset_index(drop=True)
    assert old_p["p"].equals(new_p["p"])
    p_cols = [c for c in old_p.columns if c != "p"]
    print("maximum absolute difference among p-summary statistics:",
          np.max(np.abs(new_p[p_cols].to_numpy() - old_p[p_cols].to_numpy())))

    comparison = old[["seed", "p", "iterations", "mu_pred", "mu_obs"]].copy()
    comparison = comparison.rename(columns={
        "iterations": "iterations_released",
        "mu_pred": "mu_pred_released",
        "mu_obs": "mu_obs_released",
    })
    comparison["iterations_rerun"] = new["iterations"]
    comparison["mu_pred_rerun"] = new["mu_pred"]
    comparison["mu_obs_rerun"] = new["mu_obs"]
    comparison["mu_obs_abs_diff"] = np.abs(
        comparison["mu_obs_rerun"] - comparison["mu_obs_released"]
    )

    display(comparison.sort_values("mu_obs_abs_diff", ascending=False).head(10))
else:
    print("Set RERUN_EXPERIMENT = True to perform the end-to-end rerun comparison.")


## Reproduce the main A0 figure

In [ ]:
# Use the fresh rerun by default; otherwise use the released original results.
if RERUN_EXPERIMENT:
    results_df = rerun_summary.copy()
    figure_source = "fresh rerun"
else:
    results_df = released_summary.copy()
    figure_source = "released original results"

print("plotting from:", figure_source)

results_df["signed_rel_err_pct"] = (
    100 * (results_df["mu_obs"] - results_df["mu_pred"]) / results_df["mu_pred"]
)

err_summary = (
    results_df.groupby("p", as_index=False)
    .agg(mean_signed_err=("signed_rel_err_pct", "mean"))
)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.92))

# Panel (a): predicted versus observed contraction factor
ax = axes[0]
ax.scatter(
    results_df["mu_pred"],
    results_df["mu_obs"],
    s=20,
    alpha=0.55,
    label="Primary-window runs",
)

lo = min(results_df["mu_pred"].min(), results_df["mu_obs"].min())
hi = max(results_df["mu_pred"].max(), results_df["mu_obs"].max())
pad = 0.02 * (hi - lo)

ax.plot(
    [lo - pad, hi + pad],
    [lo - pad, hi + pad],
    linestyle="--",
    linewidth=1,
    color="black",
    label=r"$y=x$",
)
ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(lo - pad, hi + pad)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel(r"Predicted contraction $\mu_{\rm pred}$")
ax.set_ylabel(r"Observed contraction $\mu_{\rm obs}$")
ax.legend(frameon=False)
ax.grid(alpha=0.2)

# Panel (b): signed relative prediction error
ax = axes[1]
rng_jitter = np.random.default_rng(0)

for j, p_value in enumerate(p_grid):
    sub = results_df[results_df["p"] == p_value]
    jitter = rng_jitter.uniform(-0.6, 0.6, size=len(sub))
    ax.scatter(
        p_value + jitter,
        sub["signed_rel_err_pct"],
        s=18,
        alpha=0.30,
        color="lightskyblue",
        label="Individual runs" if j == 0 else None,
    )

ax.plot(
    err_summary["p"],
    err_summary["mean_signed_err"],
    marker="o",
    linewidth=1.5,
    color="black",
    label="Mean signed error",
)
ax.axhline(0, linestyle="--", linewidth=1, color="black")
ax.set_xlabel(r"$p$")
ax.set_ylabel(r"$100(\mu_{\rm obs}-\mu_{\rm pred})/\mu_{\rm pred}$ (%)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)

fig.tight_layout()

figure_path = OUT_DIR / "A0_main_agreement_figure.pdf"
fig.savefig(figure_path, bbox_inches="tight")
plt.show()

print("saved:", figure_path)


## A few checks reported in the paper

In [ ]:
bad_primary = results_df[results_df["fit_R2"] < 0.99]

print("runs:", len(results_df))
print("primary-window fits with R^2 >= 0.99:", len(results_df) - len(bad_primary), "/", len(results_df))
print("primary-window fits with R^2 < 0.99:")
display(bad_primary[["seed", "p", "fit_R2", "mu_pred", "mu_obs"]])

abs_rel_pct = 100 * np.abs(results_df["mu_obs"] - results_df["mu_pred"]) / results_df["mu_pred"]
print("median absolute percentage error:", np.median(abs_rel_pct))
print("mean absolute percentage error:", np.mean(abs_rel_pct))
print("maximum absolute percentage error:", np.max(abs_rel_pct))